MLflow setup:
* Tracking server: GCP VM
* Backend store: GCP Cloud SQL postgresql database
* Artifacts store: GCP bucket

The experiments can be explored by accessing the remote server.

In [3]:
import mlflow
import os

# Jupyter lab server is currently running on the same VM as the MLFlow
# but MLFlow server endpoint is also served by Tailscale and available to all devices in the tailnet.

# TRACKING_SERVER_HOST = "mlops-vm.tailXXXXX.ts.net"

# In this notebook we will use VM's local net to access the MLFlow server.

TRACKING_SERVER_HOST = "127.0.0.1"
mlflow.set_tracking_uri(f"http://{TRACKING_SERVER_HOST}:5000")

In [4]:
mlflow.search_experiments()

[<Experiment: artifact_location='mlflow-artifacts:/2', creation_time=1786723449402, experiment_id='2', last_update_time=1786723449402, lifecycle_stage='active', name='new-experiment', tags={}>,
 <Experiment: artifact_location='gs://mlops-bucket-hashan-224322/1', creation_time=1786664617927, experiment_id='1', last_update_time=1786664617927, lifecycle_stage='active', name='nyc-taxi-experiment', tags={}>,
 <Experiment: artifact_location='gs://$GCP_BUCKET/0', creation_time=1786645869124, experiment_id='0', last_update_time=1786645869124, lifecycle_stage='active', name='Default', tags={}>]

In [7]:
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score

mlflow.set_experiment("new-experiment")

with mlflow.start_run():

    X, y = load_iris(return_X_y=True)

    params = {"C": 0.1, "random_state": 42}
    mlflow.log_params(params)

    lr = LogisticRegression(**params).fit(X, y)
    y_pred = lr.predict(X)
    mlflow.log_metric("accuracy", accuracy_score(y, y_pred))

    mlflow.sklearn.log_model(lr, artifact_path="models")
    print(f"default artifacts URI: '{mlflow.get_artifact_uri()}'")

2026/08/14 16:19:56 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/08/14 16:20:00 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


default artifacts URI: 'mlflow-artifacts:/2/f673250d47b34bea9ad2e496aceff937/artifacts'
🏃 View run delicate-bird-754 at: http://127.0.0.1:5000/#/experiments/2/runs/f673250d47b34bea9ad2e496aceff937
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


In [9]:
from mlflow.tracking import MlflowClient


client = MlflowClient(f"http://{TRACKING_SERVER_HOST}:5000")

In [10]:
run_id = client.search_runs(experiment_ids=['2'])[0].info.run_id
mlflow.register_model(
    model_uri=f"runs:/{run_id}/models",
    name='iris-classifier'
)

Successfully registered model 'iris-classifier'.
2026/08/14 16:21:46 WARNING mlflow.tracking._model_registry.fluent: Run with id f673250d47b34bea9ad2e496aceff937 has no artifacts at artifact path 'models', registering model based on models:/m-cf4b3533965e4067a1c7b7a45067bcd2 instead
2026/08/14 16:21:46 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: iris-classifier, version 1
Created version '1' of model 'iris-classifier'.


<ModelVersion: aliases=[], creation_timestamp=1786724506571, current_stage='None', deployment_job_state=<ModelVersionDeploymentJobState: current_task_name='', job_id='', job_state='DEPLOYMENT_JOB_CONNECTION_STATE_UNSPECIFIED', run_id='', run_state='DEPLOYMENT_JOB_RUN_STATE_UNSPECIFIED'>, description='', last_updated_timestamp=1786724506571, metrics=None, model_id=None, name='iris-classifier', params=None, run_id='f673250d47b34bea9ad2e496aceff937', run_link='', source='models:/m-cf4b3533965e4067a1c7b7a45067bcd2', status='READY', status_message=None, tags={}, user_id='', version='1'>